# P5 — Hash-chained audit log and explainability

A walk-through of P5's modules:

- `sentinel.audit.chain`: append-only log with SHA-256 chaining (`HashChainAuditSink`).
- `sentinel.narrative`: structured alert facts, deterministic text, validation of model-written
  text, and fallback.

**What is mocked here.** The forecaster and the detector are the backend's mocks
(`mock-persistence`, `mock-threshold`), and the data are synthetic (`sentinel.synthetic`). The
"full" alert (with region, CUSUM and contributions) is invented (`sentinel.narrative.mock`).
None of this is a leak detection. In section 4 the "LLM" is a fake function. Section 5 calls
Granite on watsonx.ai, but only if the `WATSONX_*` variables are set.

In [ ]:
import json
import os
import shutil
import sys
import tempfile
from datetime import UTC, datetime
from pathlib import Path

# On the VM the clone is ~/shared/civicwater/users/<name>/repo. Walk up from the notebook's
# folder to the repository's pyproject.toml, so the same cell works locally and on the VM.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO / "src"))  # import YOUR clone

for v in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[v] = "1"

%load_ext autoreload
%autoreload 2

import sentinel

print("sentinel from:", Path(sentinel.__file__).parent)

In [ ]:
from sentinel.audit import HashChainAuditSink, audit_path_for, verify_alert, verify_file
from sentinel.config import Settings
from sentinel.main import build_pipeline
from sentinel.narrative import FunctionNarrator, WatsonxNarrator, build_prompt, explain
from sentinel.narrative.mock import mock_alert
from sentinel.schemas.alerts import AuditEvent
from sentinel.schemas.windows import SensorWindow
from sentinel.sensors import SensorRegistry
from sentinel.synthetic import generate

# The demo's files go to a temporary folder, outside git.
WORK = Path(tempfile.mkdtemp(prefix="p5-audit-"))
print("files in:", WORK)


def show(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))

## 1. The chain on its own

Each event becomes one JSON line with `seq`, `payload_sha256`, `prev_hash` and `hash`:

`hash = SHA-256(prev_hash ‖ canonical JSON of the record without prev_hash and hash)`

In [ ]:
sink = HashChainAuditSink(WORK / "demo.jsonl")
for i in range(3):
    ref = sink.append(
        AuditEvent(
            type="window.scored",
            window_end=datetime(2019, 1, 1, 0, 5 * i, tzinfo=UTC),
            payload={"status": "scored", "i": i},
        )
    )
    print(ref)

show(sink.records()[1])
print(sink.verify())
print("head (keep it outside the VM):", sink.head)

## 2. Tampering

Changing, removing or reordering a record breaks the chain from that record on. Rewriting the
whole file consistently is only caught by comparing with the head kept elsewhere (the anchor).

In [ ]:
anchor = sink.head
lines = (WORK / "demo.jsonl").read_text().splitlines()

altered = WORK / "altered.jsonl"
rec = json.loads(lines[1])
rec["payload"]["status"] = "insufficient_history"
altered.write_text("\n".join([lines[0], json.dumps(rec), lines[2]]) + "\n")
print("payload altered:", verify_file(altered))

removed = WORK / "removed.jsonl"
removed.write_text("\n".join([lines[0], lines[2]]) + "\n")
print("record removed:", verify_file(removed))

forged = HashChainAuditSink(WORK / "forged.jsonl")  # an attacker recomputes every hash
for i in range(3):
    forged.append(
        AuditEvent(
            type="window.scored",
            window_end=datetime(2019, 1, 1, 0, 5 * i, tzinfo=UTC),
            payload={"status": "forged", "i": i},
        )
    )
print("consistent rewrite, no anchor:", verify_file(WORK / "forged.jsonl").ok)
print("consistent rewrite, with anchor:", verify_file(WORK / "forged.jsonl", anchor))

## 3. Plugged into the backend pipeline (mocks)

One synthetic day (288 windows) with a pressure step on 3 sensors from window 200. The backend
calls `sink.append(...)` for every window and every alert; here we only swap the sink.

In a real replay on the VM the file goes to the agreed place, outside the git clones:
`audit_path_for("<run>")` → `~/shared/civicwater/audit/<run>.jsonl`. The demo uses the temporary
folder.

In [ ]:
print("agreed location on the VM:", audit_path_for("replay-2019"))

settings = Settings()  # 288 / 144 / 300 s
registry = SensorRegistry.load(REPO / "config" / "sensors.provisional.json")
chain = HashChainAuditSink(audit_path_for("demo-pipeline", root=WORK))
pipeline = build_pipeline(settings, registry=registry, audit_sink=chain)

for payload in generate(
    datetime(2019, 3, 14, tzinfo=UTC), 288, registry,
    seed=7, noise_m=0.05, leak_at=200, leak_sensors=registry.pressure[:3],
):
    pipeline.ingest(SensorWindow.model_validate(payload))

In [ ]:
# Take the first alert from the chain's own alert.raised record.
raised = [r for r in chain.records() if r["type"] == "alert.raised"]
print("events in the chain:", len(chain.records()), "| alerts:", len(raised))
alert = pipeline.alerts.get(raised[0]["alert_id"])
print("alert:", alert.alert_id, "| audit:", alert.audit)
print("chain:", chain.verify())
print("alert matches its record:", verify_alert(alert, chain.get(alert.audit.seq)) is None)

## 4. Explanation

Without a model: the deterministic text, always available. With a model: the text is accepted
only if every sensor, region, number, date and time in it comes from the alert payload and sits
in the right place; otherwise the deterministic text comes back. Explanations are in English.

In [ ]:
e = explain(alert)  # pipeline alert with mocks: no region and no CUSUM
print(e.text)
print("\nsource:", e.source, "| payload_sha256 == record:",
      e.alert_payload_sha256 == chain.get(alert.audit.seq)["payload_sha256"])

In [ ]:
rich = mock_alert()  # invented: region, CUSUM, onset and contributions
print(explain(rich).text)
print("\n--- prompt sent to the LLM ---\n")
print(build_prompt(rich))

In [ ]:
# Fake "LLMs": one faithful text, one that invents a sensor, a region, a number and a time.
good = FunctionNarrator("fake-llm@0", lambda prompt: (
    "[SIMULATION] Alert A-20190314T1230Z-1 reports a suspected leak in region R4. "
    "Sensors P14 and P21 read 0.91 m and 0.62 m below the expected pressure since "
    "2019-03-14 02:10 UTC. The CUSUM statistic reached 12.4 against a threshold of 8.0."
))
hallucinated = FunctionNarrator("fake-llm@0", lambda prompt: (
    "[SIMULATION] Confirmed leak in region R2: sensor P07 dropped 7.3 m at 03:15."
))

for narrator in (good, hallucinated):
    e = explain(rich, narrator)
    print("source:", e.source, "| fallback:", e.fallback_reason)
    print("check:", e.check)
    print(e.text, "\n")

## 5. Granite on watsonx.ai

`WatsonxNarrator` calls the watsonx.ai chat API with the standard library only (nothing to
install on the LinuxONE). It reads `WATSONX_APIKEY`, `WATSONX_URL` (e.g.
`https://us-south.ml.cloud.ibm.com`), `WATSONX_PROJECT_ID` and `WATSONX_MODEL_ID`. On 2026-10-17
the only Granite chat model was `ibm/granite-4-h-small`, in us-south and eu-de only (see
`docs/audit-and-narrative.md`).

**Never write the API key into the notebook or commit it.** To set it for this kernel only:

```python
import getpass
os.environ["WATSONX_APIKEY"] = getpass.getpass("WATSONX_APIKEY: ")
```

Without the variables the cell below just says so. If Granite fails or invents something,
`explain` returns the deterministic text and records why in `fallback_reason`. The Granite model
that runs on the VM itself is in `p5_linuxone_granite.ipynb`.

In [ ]:
if all(os.environ.get(k) for k in
       ("WATSONX_APIKEY", "WATSONX_URL", "WATSONX_PROJECT_ID", "WATSONX_MODEL_ID")):
    granite = WatsonxNarrator.from_env()
    for target in (rich, alert):
        e = explain(target, granite)
        print("source:", e.source, "| model:", e.generator, "| fallback:", e.fallback_reason)
        print("check:", e.check)
        print(e.text, "\n")
else:
    print("WATSONX_* variables not set: real Granite call skipped.")

## 6. External anchor and command-line verification

At the end of a replay, write the head down outside the VM and verify against it later:

```bash
python -m sentinel.audit.verify <file.jsonl> --anchor-seq <seq> --anchor-hash <hash>
```

In [ ]:
print("head:", chain.head)
print(f"python -m sentinel.audit.verify {chain.path} --anchor-seq {chain.head.seq} --anchor-hash {chain.head.hash}")

In [ ]:
shutil.rmtree(WORK)  # remove the demo's files